## 1. Install required Python packages

In [1]:
%pip install langchain langgraph langchain-openai

Note: you may need to restart the kernel to use updated packages.


## 2. Get Endpoints

Retrieve the FQDN of the self-hosted LLM and the Cosmos DB connection details from Terraform outputs.

In [2]:
aca_gemma4_31b_it_a100_fqdn = ! terraform -chdir=infra output -raw aca_gemma4_31b_it_a100_fqdn
aca_gemma4_31b_it_a100_fqdn = aca_gemma4_31b_it_a100_fqdn.n
print("LLM Endpoint:", aca_gemma4_31b_it_a100_fqdn)

aca_qwen_36_35b_a100_fqdn = ! terraform -chdir=infra output -raw aca_qwen_36_35b_a100_fqdn
aca_qwen_36_35b_a100_fqdn = aca_qwen_36_35b_a100_fqdn.n
print("LLM Endpoint:", aca_qwen_36_35b_a100_fqdn)

foundry_endpoint = ! terraform -chdir=infra output -raw foundry_endpoint
foundry_endpoint = foundry_endpoint.n
print("Foundry Endpoint:", foundry_endpoint)

foundry_api_key = ! terraform -chdir=infra output -raw foundry_api_key
foundry_api_key = foundry_api_key.n
print("Foundry API Key:", f"{foundry_api_key[-10:]}...")  # Print only the last 10 characters for security

llm_model_deployment_name_chatgpt = ! terraform -chdir=infra output -raw llm_model_deployment_name_chatgpt
llm_model_deployment_name_chatgpt = llm_model_deployment_name_chatgpt.n
print("LLM Model Deployment Name (ChatGPT):", llm_model_deployment_name_chatgpt)

LLM Endpoint: ╷
│ Error: Output "aca_gemma4_31b_it_a100_fqdn" not found
│ 
│ The output variable requested could not be found in the state file. If you
│ recently added this to your configuration, be sure to run `terraform
│ apply`, since the state won't be updated with new output variables until
│ that command is run.
╵
LLM Endpoint: qwen-3-6-35b-a100.gentlemushroom-793350b5.swedencentral.azurecontainerapps.io
Foundry Endpoint: https://foundry-555.cognitiveservices.azure.com/
Foundry API Key: AAACOGxtMj...
LLM Model Deployment Name (ChatGPT): gpt-5.4


## 3. Set Up the LLM Model

Create a `ChatOpenAI` model pointing at the vLLM-compatible endpoint running on Azure Container Apps.

In [3]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    base_url=f"http://{aca_qwen_36_35b_a100_fqdn}/v1",
    api_key="EMPTY",
    model="Qwen/Qwen3.6-35B-A3B",
    streaming=True,
    max_completion_tokens=512
)

# model = ChatOpenAI(
#     base_url=f"http://{aca_gemma4_31b_it_a100_fqdn}/v1",
#     api_key="EMPTY",
#     model="google/gemma-4-31B-it",
#     streaming=True,
#     max_completion_tokens=512
# )

# model = ChatOpenAI(
#     base_url=f"{foundry_endpoint}/openai/v1",
#     api_key=foundry_api_key,
#     model=llm_model_deployment_name_chatgpt,
#     streaming=True,
#     max_completion_tokens=512
# )

## 4. Test the Model

Invoke the model with a test prompt to ensure it's working correctly.

In [6]:
from langchain_core.messages import HumanMessage

response = model.stream([HumanMessage(content="Tell me about yourself.")])

for chunk in response:
    print(chunk.content, end="", flush=True)



I’m Qwen, a large language model developed by Alibaba Group’s Tongyi Lab. I’m designed to be a helpful, clear, and thoughtful AI assistant that can adapt to a wide range of tasks—like answering questions, solving problems,

## 5. Use the Model in a LangChain Agent

In [7]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

agent = create_agent(
    model=model,
    tools=[],
    middleware=[],
    checkpointer=None,
)

response = agent.stream({"messages": HumanMessage(content="Tell me about yourself")})

async for step in agent.astream(
    {"messages": [HumanMessage(content="Tell me about yourself")]},
    stream_mode="values"
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Tell me about yourself
================================== Ai Message ==================================



I’m Qwen, a large language model developed by Alibaba Group’s Tongyi Lab. I’m designed to be a clear, honest, and practical thinking partner—helping with writing, coding, analysis, problem-solving, research, and more. I support many languages, work with long documents and complex files, and can adapt to different workflows and formats. 

What are you working on? I’d be glad to help
